In [1]:
import pandas as pd 

df = pd.read_csv('Dataset1.csv')


print("---current shape of csv file---",df.shape)
df.info()



---current shape of csv file--- (186074, 12)
<class 'pandas.DataFrame'>
RangeIndex: 186074 entries, 0 to 186073
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   SN              186074 non-null  int64
 1   Train_No        186074 non-null  int64
 2   Station_Code    186074 non-null  str  
 3   1A              186074 non-null  int64
 4   2A              186074 non-null  int64
 5   3A              186074 non-null  int64
 6   SL              186074 non-null  int64
 7   Station_Name    186074 non-null  str  
 8   Route_Number    186074 non-null  int64
 9   Arrival_time    186074 non-null  str  
 10  Departure_Time  186074 non-null  str  
 11  Distance        186074 non-null  int64
dtypes: int64(8), str(4)
memory usage: 17.0 MB


In [2]:
import numpy as np

#missing values chek krenge dataset me se (janeraly missing values vorigine station me arrival time
#none hota hay and destinatio station me departure time nano hota hay)

print ("missing values count per column befor handling ")
print(df.isnull().sum())

# Agar starting station par Arrival missing/None ho, toh use Departure_Time se fill kar sakte hain
#  ya 'Source' mark kar sakte hain
# Similarly, Destination station par Departure missing ho toh Arrival_time se fill kar sakte  hain
df['Arrival_time'] = df['Arrival_time'].replace(['None', 'none', ''], np.nan)
df['Departure_Time'] = df['Departure_Time'].replace(['None', 'none', ''], np.nan)

#abhi bhi agar kuch missing values ho to unhe check kar lenge
remaining_null_values = df[['Arrival_time', 'Departure_Time']].isnull().sum()
print("\n Remaining missing null values :")
print (remaining_null_values)



missing values count per column befor handling 
SN                0
Train_No          0
Station_Code      0
1A                0
2A                0
3A                0
SL                0
Station_Name      0
Route_Number      0
Arrival_time      0
Departure_Time    0
Distance          0
dtype: int64

 Remaining missing null values :
Arrival_time      0
Departure_Time    0
dtype: int64


In [5]:
#Duplicate records check karenge 
duplicate_records = df.duplicated().sum()
print(f"\n Duplicate records count is: {duplicate_records}")


 Duplicate records count is: 0


In [6]:
df.head(5)

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance
0,1,107,SWV,100,100,100,100,SAWANTWADI R,1,00:00:00,10:25:00,0
1,2,107,THVM,260,228,196,164,THIVIM,1,11:06:00,11:08:00,32
2,3,107,KRMI,345,296,247,198,KARMALI,1,11:28:00,11:30:00,49
3,4,107,MAO,490,412,334,256,MADGOAN JN.,1,12:10:00,00:00:00,78
4,1,108,MAO,100,100,100,100,MADGOAN JN.,1,00:00:00,20:30:00,0


In [8]:
#pahle distinict-unique records select karenge duplicates ko drop karenge 
df_cleaned = df.drop_duplicates().reset_index(drop=True)

#fhir bhi agar kisi record ka duplicate record rah gaya to use baki subset columns ke help se remove kardenge
df_cleaned = df_cleaned.drop_duplicates(subset=['Train_No', 'Station_Code', 'SN']).reset_index(drop = True)

print(f"shape after removing duplicates: {df_cleaned.shape}")


shape after removing duplicates: (186074, 12)


In [10]:
#task 3.3 corect station sequence check karenge 
#sort karenge train number and sn ke hisab se
df_cleaned= df_cleaned.sort_values(by= ['Train_No', 'SN']).reset_index(drop = True)

# Check karna ki kya distance har train me lagatar  increase ho raha hai
df_cleaned['Distance'] = pd.to_numeric(df_cleaned['Distance'], errors='coerce')

# Har train ke andar check karenge ki previous distance agle distance se chota ya barabar ho
df_cleaned['Distance_Diff'] = df_cleaned.groupby('Train_No')['Distance'].diff()

# Out-of-order stations dekhna (jahan difference negative ho, matlab achanak distance kam ho gaya)
order_issues = df_cleaned[df_cleaned['Distance_Diff'] < 0]

print(f"Number of station sequence anomalies found: {len(order_issues)}")

if len(order_issues) == 0:
    print("Verification Passed: All station routes are strictly in correct sequential order!")
else:
    print("Warning: Found stations with incorrect order:")
    display(order_issues[['Train_No', 'SN', 'Station_Code', 'Distance', 'Distance_Diff']])

# Temporary column drop kar denge verification ke baad
df_cleaned.drop(columns=['Distance_Diff'], inplace=True)

Number of station sequence anomalies found: 0
Verification Passed: All station routes are strictly in correct sequential order!


In [11]:
#final cleaned file save karenge 
output_file_name = 'Cleaned_train_dataset.csv'
#.to_csv() se csv file me save karenge 
df_cleaned.to_csv(output_file_name, index=False)
print (f"\n successfuly: cleaned dataset seved as : {output_file_name}")
print(f" final cleaned dataset structure: {df_cleaned.shape}")



 successfuly: cleaned dataset seved as : Cleaned_train_dataset.csv
 final cleaned dataset structure: (186074, 12)
